# Model fit indices
print("\n=== Model Fit Indices ===")
stats = semopy.calc_stats(model)
print(stats.to_string())


In [ ]:

> **Interpreting the Output:** The **Path Coefficients** table lists every estimated relationship. The `Estimate` column gives the standardized path coefficient; `p-value` tests $H_0$: path = 0. Focus on the **`Sat ~ Brand`** and **`Sat ~ Func`** rows to see which antecedents drive Satisfaction, and the **`Loyal ~ Sat`** row to confirm the mediating role. For **Model Fit Indices**: **CFI > 0.95** indicates excellent fit (values above 0.90 are acceptable); **RMSEA < 0.08** indicates acceptable fit (< 0.05 is excellent); **SRMR < 0.08** confirms the model reproduces the observed correlations well. A well-fitting SEM justifies interpreting the structural path estimates as theoretically meaningful.

### Student Task
Create `StudentName_SEM.py` in Visual Studio Code. Extend the measurement model to include `Usab =~ Usab1 + Usab2` and `Price =~ Price1 + Price2`. Add `Usab` and `Price` as additional predictors of `Sat` in the structural model. Re-fit with `semopy` and report the path coefficients for all four predictors of Satisfaction. Which construct has the largest standardized path coefficient? Check CFI and RMSEA to confirm model fit.

### Evaluation Questions
1. What two sub-models does SEM combine, and what does each assess?
2. Why are latent variables preferred over simple sum scores in SEM?
3. What does CFI $> 0.95$ indicate about a model?
4. Why does SEM produce more accurate path coefficients than standard regression?
5. In the Gogoro SEM, what is the hypothesized role of Customer Satisfaction?

## Section 4: Integrated Model Testing

### Objective
To combine mediation and moderation in a unified framework

**Keywords:** moderated mediation, Index of Moderated Mediation (IMM), conditional indirect effect, interaction term, integrated model, Hayes PROCESS — testing moderated mediation — and evaluate whether the indirect effect of Brand Image on Loyalty through Satisfaction varies across levels of a moderating variable.

### Statistical Perspective
**Moderated Mediation** examines whether the strength of a mediated pathway ($IV \xrightarrow{a} M \xrightarrow{b} DV$) depends on a moderating variable $W$.

**The Integrated Framework:**

$$M = a_1(IV) + a_2(W) + a_3(IV \times W) + \epsilon_1$$
$$DV = b(M) + c'(IV) + \epsilon_2$$

The **conditional indirect effect** at a given level of $W$ is:

$$\theta(W) = (a_1 + a_3 \cdot W) \times b$$

**Hypothesis Tests:**

| Test | $H_0$ | $H_1$ |
|------|---------|---------|
| Moderated path ($a_3$) | $a_3 = 0$: Price does not moderate Brand → Satisfaction | $a_3 \neq 0$: The indirect pathway varies by Price level |
| Mediator path ($b$) | $b = 0$: Satisfaction does not predict Loyalty | $b \neq 0$: Satisfaction significantly drives Loyalty |
| IMM = $a_3 \times b$ | $IMM = 0$: No moderated mediation | Bootstrapped 95% CI excludes zero: Confirmed |

- A significant $a_3$ ($p < 0.05$) confirms the indirect effect differs across levels of $W$.
- **Index of Moderated Mediation (IMM):** $a_3 \times b$. If its 95% bootstrapped CI excludes zero, moderated mediation is confirmed.

**Why It Matters:**
Brand Image may increase Satisfaction (mediated path), but this effect may be stronger for customers who are highly price-sensitive (moderated by Price perception). Integrated testing captures this complexity in a single testable model.

**Industry Application:** A streaming service finds that content quality (IV) boosts subscriber retention (DV) through perceived value (Mediator), but this indirect path is significantly stronger for price-sensitive users (Moderator) — directing targeted pricing strategies to high-risk churn segments.

### Python Example Code
```{python}
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('gogoro_data.csv')

df['Brand_c'] = df['Brand1'] - df['Brand1'].mean()
df['Price_c'] = df['Price1'] - df['Price1'].mean()
df['Sat_c']   = df['Sat1']   - df['Sat1'].mean()
df['Brand_x_Price'] = df['Brand_c'] * df['Price_c']

Xa = sm.add_constant(df[['Brand_c', 'Price_c', 'Brand_x_Price']])
ma = sm.OLS(df['Sat1'], Xa).fit()

Xb = sm.add_constant(df[['Sat_c', 'Brand_c']])
mb = sm.OLS(df['Loyal1'], Xb).fit()

a3  = ma.params['Brand_x_Price']
b   = mb.params['Sat_c']
IMM = a3 * b

print(f"Interaction (a3): {a3:.3f}, p={ma.pvalues['Brand_x_Price']:.3f}")
print(f"Path b (Sat->Loyal): {b:.3f}")
print(f"Index of Moderated Mediation (IMM): {IMM:.3f}")



> **Interpreting the Output:** **Line 1 — Interaction ($a_3$):** The coefficient on `Brand_x_Price` in the Satisfaction equation. If $p < 0.05$, $H_0$ for the moderated path is rejected — Price significantly moderates how Brand Image drives Satisfaction. **Line 2 — Path $b$:** The effect of mean-centered Satisfaction on Loyalty; a significant value confirms the mediating link. **Line 3 — IMM:** The product $a_3 \times b$ quantifies moderated mediation. A positive IMM indicates the indirect Brand → Sat → Loyal path is stronger at higher Price levels; a negative IMM indicates it weakens. Formal significance of the IMM requires bootstrapped confidence intervals (95% CI excluding zero = confirmed moderated mediation).

### Student Task
Create `StudentName_IntegratedModel.py` in Visual Studio Code. Test whether `Usab1` moderates the mediated path from `Func1` (IV) through `Sat1` (Mediator) to `Loyal1` (DV). Report the Index of Moderated Mediation (IMM = $a_3 \times b$) and interpret whether the conditional indirect effect is stronger at high or low Usability.

### Evaluation Questions
1. What is moderated mediation, and how does it differ from simple mediation?
2. What is the Index of Moderated Mediation (IMM) and how is it interpreted?
3. Why must the interaction term be included in the mediator equation (path $a$)?
4. If IMM = $0.08$ with a 95% CI of $[0.02, 0.18]$, what is the conclusion?
5. How does integrated model testing provide more complete insights than mediation or moderation alone?

